In [154]:
import random
import pandas as pd
from datasets import Dataset
import os 
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset
import seaborn as sns
from datasets import load_dataset, load_from_disk
import numpy as np
import json

In [60]:
cd /mnt/home/al2644/research/projects/rlvr

/mnt/home/al2644/research/projects/rlvr


In [156]:
train_df = pd.read_parquet("./data/train/math8k/train.parquet")
test_df = pd.read_parquet("./data/train/math8k/test.parquet")

In [157]:
train_df

,data_source,prompt,ability,reward_model,extra_info
0,math8k,[{'content': 'Let $a$ and $b$ be the two real ...,math,"{'ground_truth': '118', 'style': 'rule'}","{'difficulty': 5, 'index': 0, 'split': 'train'..."
1,math8k,[{'content': 'For how many integer values of $...,math,"{'ground_truth': '5', 'style': 'rule'}","{'difficulty': 5, 'index': 1, 'split': 'train'..."
2,math8k,[{'content': 'A car is averaging 50 miles per ...,math,"{'ground_truth': '30', 'style': 'rule'}","{'difficulty': 3, 'index': 2, 'split': 'train'..."
3,math8k,[{'content': 'Find the greatest common divisor...,math,"{'ground_truth': '1', 'style': 'rule'}","{'difficulty': 3, 'index': 3, 'split': 'train'..."
4,math8k,[{'content': 'How many ounces of pure water mu...,math,"{'ground_truth': '15', 'style': 'rule'}","{'difficulty': 3, 'index': 4, 'split': 'train'..."
...,...,...,...,...,...
7047,math8k,[{'content': 'If three coins are tossed at the...,math,"{'ground_truth': '\frac{3}{8}', 'style': 'rule'}","{'difficulty': 3, 'index': 8516, 'split': 'tra..."
7048,math8k,[{'content': 'If the odds for pulling a prize ...,math,"{'ground_truth': '\frac{4}{7}', 'style': 'rule'}","{'difficulty': 3, 'index': 8517, 'split': 'tra..."
7049,math8k,[{'content': 'How many numbers in the list $43...,math,"{'ground_truth': '1', 'style': 'rule'}","{'difficulty': 3, 'index': 8519, 'split': 'tra..."
7050,math8k,[{'content': 'A car travels 40 kph for 20 kilo...,math,"{'ground_truth': '51', 'style': 'rule'}","{'difficulty': 5, 'index': 8521, 'split': 'tra..."


In [158]:
test_df

,data_source,prompt,ability,reward_model,extra_info
0,aime 24,"[{'content': 'Let $x,y$ and $z$ be positive re...",math,"{'ground_truth': '33', 'style': 'rule'}","{'difficulty': 'aime 24', 'index': 0, 'split':..."
1,aime 24,"[{'content': 'Let $O(0,0), A(\tfrac{1}{2}, 0),...",math,"{'ground_truth': '23', 'style': 'rule'}","{'difficulty': 'aime 24', 'index': 1, 'split':..."
2,aime 24,[{'content': 'Jen enters a lottery by picking ...,math,"{'ground_truth': '116', 'style': 'rule'}","{'difficulty': 'aime 24', 'index': 2, 'split':..."
3,aime 24,[{'content': 'Alice and Bob play the following...,math,"{'ground_truth': '809', 'style': 'rule'}","{'difficulty': 'aime 24', 'index': 3, 'split':..."
4,aime 24,[{'content': 'Eight circles of radius $34$ are...,math,"{'ground_truth': '197', 'style': 'rule'}","{'difficulty': 'aime 24', 'index': 4, 'split':..."
...,...,...,...,...,...
795,math500,[{'content': 'What is the domain of the functi...,math,"{'ground_truth': '(2,12) \cup (12,102)', 'styl...","{'difficulty': 'math500', 'index': 795, 'split..."
796,math500,[{'content': 'Let $z = 1+i$ and $w = \dfrac{3z...,math,"{'ground_truth': '\frac{5}{13}', 'style': 'rule'}","{'difficulty': 'math500', 'index': 796, 'split..."
797,math500,[{'content': 'An equiangular octagon has four ...,math,"{'ground_truth': '\frac{7}{2}', 'style': 'rule'}","{'difficulty': 'math500', 'index': 797, 'split..."
798,math500,[{'content': 'A sequence $(a_n)$ is defined as...,math,"{'ground_truth': '-1', 'style': 'rule'}","{'difficulty': 'math500', 'index': 798, 'split..."


In [136]:
train_problem = set(train_df['prompt'].apply(lambda x: x [0]["content"]))
test_problem = set(test_df['prompt'].apply(lambda x: x [0]["content"]))

In [97]:
oct_train_df = pd.read_parquet("./data/train/octothink.math.8k/train.parquet")
oct_test_df = pd.read_parquet("./data/train/octothink.math.8k/test.parquet")

In [111]:
oct_train_problems = set(oct_train_df['prompt'].apply(lambda x: x[-1]['content']))
oct_test_problems = set(oct_test_df[oct_test_df['data_source']=='math']['prompt'].apply(lambda x: x[-1]['content']))

In [107]:
len(train_problem.difference(oct_problems))

116

In [119]:
oct_test_df['data_souce']

,data_source,prompt,ability,reward_model,extra_info
0,aime,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '204', 'stype': 'rule'}","{'id': '57e97007-1081-404f-bd5c-8c9fac363c51',..."
1,aime,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '204', 'stype': 'rule'}","{'id': '57e97007-1081-404f-bd5c-8c9fac363c51',..."
2,aime,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '204', 'stype': 'rule'}","{'id': '57e97007-1081-404f-bd5c-8c9fac363c51',..."
3,aime,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '204', 'stype': 'rule'}","{'id': '57e97007-1081-404f-bd5c-8c9fac363c51',..."
4,aime,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '113', 'stype': 'rule'}","{'id': '83384151-50bf-48e4-8e4b-6864b76b1bd3',..."
...,...,...,...,...,...
267,pad,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '2', 'stype': 'rule'}","{'id': 'c4f0bd45-95db-4881-9b81-448d9b024bf6',..."
268,pad,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '2', 'stype': 'rule'}","{'id': '3f8b324e-9a7c-45e7-b11a-b7a9b751a17a',..."
269,pad,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '2', 'stype': 'rule'}","{'id': 'd7aaf6ee-be5b-4e03-826e-965186478600',..."
270,pad,"[{'content': 'Please reason step by step, and ...",other,"{'ground_truth': '2', 'stype': 'rule'}","{'id': 'fa40763f-712a-4081-9e08-aa6444c51786',..."


In [142]:
tokenizer = AutoTokenizer.from_pretrained("aochongoliverli/Qwen2.5-3B-Zero-Base")

In [151]:
print(tokenizer.apply_chat_template(test_df['prompt'][0], tokenize=False, add_generation_prompt=True))

<|im_start|>system
Please reason step by step. Think through the problem in depth before answering. Finally, put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
\[\log_2\left({x \over yz}\right) = {1 \over 2}\]
\[\log_2\left({y \over xz}\right) = {1 \over 3}\]
\[\log_2\left({z \over xy}\right) = {1 \over 4}\]
Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.<|im_end|>
<|im_start|>assistant

